# Weighting Time Benchmark

Measures `Final Weighting Time (s)` for each strategy on each dataset,
using best params from ALS_factors=100 and KNN_k=100 results CSVs.
Models are **not** retrained — only the weighting step is timed.

In [ ]:
import ast
import time
import numpy as np
import pandas as pd
import cornac
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils.sparse import transform_dataframe_to_sparse
from implicit.evaluation import train_test_split
from weighting_strategies import (
    bm25_weight, tfidf_weight, normalized_weight,
    log_weight, log_idf_weight, power_weight,
    pmi_weight, robust_user_centric_weight, sigmoid_propensity_weight,
    power_lift_weight, robust_user_centric_weight_v2,
)

N_REPS = 5

In [ ]:
def get_weighted_matrix(matrix, strategy, params):
    weighted = matrix.copy()
    if strategy == "bm25":
        weighted = bm25_weight(weighted, K1=params.get("bm25_k1"), B=params.get("bm25_b"))
    elif strategy == "log_idf":
        weighted = log_idf_weight(weighted, alpha=params.get("conf_alpha"))
    elif strategy == "power":
        weighted = power_weight(weighted, p=params.get("power_p"))
    elif strategy == "tfidf":
        weighted = tfidf_weight(weighted)
    elif strategy == "log":
        weighted = log_weight(weighted)
    elif strategy == "normalized":
        weighted = normalized_weight(weighted)
    elif strategy == "pmi":
        weighted = pmi_weight(weighted)
    elif strategy == "robust_user_centric":
        weighted = robust_user_centric_weight(weighted)
    elif strategy == "sigmoid_propensity":
        weighted = sigmoid_propensity_weight(weighted, p=params.get("p"), beta=params.get("beta"))
    elif strategy == "power_lift":
        weighted = power_lift_weight(weighted, p=params.get("p"))
    elif strategy == "no_weighting":
        pass
    return weighted

In [ ]:
def load_best_params(csv_path, algorithm):
    """Load best params per strategy for a given algorithm from a results CSV."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df["Strategy"] = df["Strategy"].str.strip()
    df["Algorithm"] = df["Algorithm"].str.strip()
    rows = df[
        (df["Algorithm"] == algorithm) &
        (df["Strategy"] != "robust_user_centric_weight_v2")
    ]
    result = {}
    for _, row in rows.iterrows():
        try:
            params = ast.literal_eval(row["Best Params"])
        except Exception:
            params = {}
        result[row["Strategy"]] = params
    return result


def time_weighting(matrix, strategy, params, n_reps=N_REPS):
    """Return mean weighting time in seconds over n_reps runs."""
    times = []
    for _ in range(n_reps):
        start = time.time()
        get_weighted_matrix(matrix, strategy, params)
        times.append(time.time() - start)
    return float(np.mean(times))


def benchmark_dataset(dataset_name, train_val_mat, als_csv, knn_csv):
    """Benchmark weighting time for all strategies on a dataset."""
    records = []
    for algo, csv_path in [("ALS_factors=100", als_csv), ("KNN_k=100", knn_csv)]:
        best_params = load_best_params(csv_path, algo)
        for strategy, params in best_params.items():
            t = time_weighting(train_val_mat, strategy, params)
            records.append({
                "Dataset": dataset_name,
                "Algorithm": algo,
                "Strategy": strategy,
                "Final Weighting Time (s)": t,
            })
            print(f"  [{algo}] {strategy}: {t:.4f}s")
    return records

## Load Datasets

In [ ]:
# LastFM 360K
lastfm_df = (
    pd.read_csv(
        "/home/coder/projects/rec-sys-research/data/lastfm-dataset-360K/usersha1-artmbid-artname-plays.tsv",
        sep="\t",
        header=None,
        usecols=[0, 2, 3],
        names=["user_id", "item_id", "play_count"],
    )
    .loc[:, ["user_id", "item_id", "play_count"]]
    .dropna()
    .rename(columns={"play_count": "target"})
)
mat, _, _ = transform_dataframe_to_sparse(lastfm_df, row_field="user_id", col_field="item_id", data_field="target")
train_val_lastfm, _ = train_test_split(mat, train_percentage=0.9, random_state=42)
print(f"LastFM 360K  train_val shape: {train_val_lastfm.shape}")

In [ ]:
# MovieLens 20M
movielens_20m_df = (
    pd.DataFrame(
        data=cornac.datasets.movielens.load_feedback(variant="20M"),
        columns=["user_id", "item_id", "target"],
    )
    .loc[:, ["user_id", "item_id", "target"]]
    .dropna()
)
mat, _, _ = transform_dataframe_to_sparse(movielens_20m_df, row_field="user_id", col_field="item_id", data_field="target")
train_val_ml20m, _ = train_test_split(mat, train_percentage=0.9, random_state=42)
print(f"MovieLens 20M  train_val shape: {train_val_ml20m.shape}")

In [ ]:
# MovieLens 100K
movielens_100k_df = (
    pd.DataFrame(
        data=cornac.datasets.movielens.load_feedback(variant="100K"),
        columns=["user_id", "item_id", "target"],
    )
    .loc[:, ["user_id", "item_id", "target"]]
    .dropna()
)
mat, _, _ = transform_dataframe_to_sparse(movielens_100k_df, row_field="user_id", col_field="item_id", data_field="target")
train_val_ml100k, _ = train_test_split(mat, train_percentage=0.9, random_state=42)
print(f"MovieLens 100K  train_val shape: {train_val_ml100k.shape}")

In [ ]:
# Steam
steam_df = (
    pd.read_csv(
        "/home/coder/projects/rec-sys-research/data/steam/steam_recommendations.csv",
        usecols=["user_id", "app_id", "hours"],
    )
    .loc[:, ["user_id", "app_id", "hours"]]
    .drop_duplicates()
    .dropna()
    .rename(columns={"app_id": "item_id", "hours": "target"})
)
mat, _, _ = transform_dataframe_to_sparse(steam_df, row_field="user_id", col_field="item_id", data_field="target")
train_val_steam, _ = train_test_split(mat, train_percentage=0.9, random_state=42)
print(f"Steam  train_val shape: {train_val_steam.shape}")

In [ ]:
# Taste Profile
taste_profile_df = pd.read_table(
    "/home/coder/projects/rec-sys-research/data/The Echo Nest Taste Profile Subset.txt",
    sep="\t",
    header=None,
    usecols=[0, 1, 2],
    names=["user_id", "item_id", "target"],
)
mat, _, _ = transform_dataframe_to_sparse(taste_profile_df, row_field="user_id", col_field="item_id", data_field="target")
train_val_taste, _ = train_test_split(mat, train_percentage=0.9, random_state=42)
print(f"Taste Profile  train_val shape: {train_val_taste.shape}")

## Run Benchmark

In [ ]:
all_records = []

datasets = [
    ("lastfm_360k",    train_val_lastfm,  "lastfm_360k_als_results.csv",    "lastfm_360k_knn_results.csv"),
    ("movielens_20m",  train_val_ml20m,   "movielens_20m_als_results.csv",  "movielens_20m_knn_results.csv"),
    ("movielens_100k", train_val_ml100k,  "movielens_100k_als_results.csv", "movielens_100k_knn_results.csv"),
    ("steam",          train_val_steam,   "steam_als_results.csv",          "steam_knn_results.csv"),
    ("taste_profile",  train_val_taste,   "taste_profile_als_results.csv",  "taste_profile_knn_results.csv"),
]

for dataset_name, train_val_mat, als_csv, knn_csv in datasets:
    print(f"\n=== {dataset_name} ===")
    records = benchmark_dataset(dataset_name, train_val_mat, als_csv, knn_csv)
    all_records.extend(records)

results_df = pd.DataFrame(all_records)
results_df

In [ ]:
results_df.to_csv("weighting_time_results.csv", index=False)
print("Saved to weighting_time_results.csv")